In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Ladda dataset som DataFrame för läsbarhet
iris = load_iris(as_frame=True)

# X = features (inputvariabler), y = target (klass i iris)
X = iris.data
y = iris.target

# Vi gör train/test-split så vi kan prata om "fit på train" och "apply på test"
# stratify=y gör att klassfördelningen blir liknande i train och test (viktigt vid klassificering)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)

(120, 4) (30, 4)


In [5]:
scaler = StandardScaler()

scaler.fit(X_train)

# learned params
print("mean_:", np.round(scaler.mean_, 3))
print("scale_:", np.round(scaler.scale_, 3))

# transform
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled train mean (feature 0):", np.round(X_train_scaled[:,0].mean(),3))

mean_: [5.842 3.048 3.77  1.205]
scale_: [0.837 0.447 1.761 0.759]
Scaled train mean (feature 0): -0.0


In [9]:
scaler.get_params()

{'copy': True, 'with_mean': True, 'with_std': True}

In [7]:
model = LogisticRegression(max_iter=200)

model.fit(X_train_scaled, y_train)

# learned params
print("coef_ shape:", model.coef_.shape)
print("intercept_:", np.round(model.intercept_,3))

# predict
pred = model.predict(X_test_scaled)

# predict_proba
proba = model.predict_proba(X_test_scaled)

print("Pred (första 5):", pred[:5])
print("Proba (första raden):", np.round(proba[0],3))

coef_ shape: (3, 4)
intercept_: [-0.306  1.909 -1.603]
Pred (första 5): [0 2 1 1 0]
Proba (första raden): [0.979 0.021 0.   ]


In [8]:
model.get_params()

{'C': 1.0,
 'class_weight': None,
 'dual': False,
 'fit_intercept': True,
 'intercept_scaling': 1,
 'l1_ratio': 0.0,
 'max_iter': 200,
 'n_jobs': None,
 'penalty': 'deprecated',
 'random_state': None,
 'solver': 'lbfgs',
 'tol': 0.0001,
 'verbose': 0,
 'warm_start': False}

In [10]:
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=200))
])

pipe.fit(X_train, y_train)

pred_pipe = pipe.predict(X_test)
print("Pred from pipeline (first 5):", pred_pipe[:5])

Pred from pipeline (first 5): [0 2 1 1 0]


In [ ]:
pipe.get_params()

{'memory': None,
 'steps': [('scaler', StandardScaler()),
  ('model', LogisticRegression(max_iter=200))],
 'transform_input': None,
 'verbose': False,
 'scaler': StandardScaler(),
 'model': LogisticRegression(max_iter=200),
 'scaler__copy': True,
 'scaler__with_mean': True,
 'scaler__with_std': True,
 'model__C': 1.0,
 'model__class_weight': None,
 'model__dual': False,
 'model__fit_intercept': True,
 'model__intercept_scaling': 1,
 'model__l1_ratio': 0.0,
 'model__max_iter': 200,
 'model__n_jobs': None,
 'model__penalty': 'deprecated',
 'model__random_state': None,
 'model__solver': 'lbfgs',
 'model__tol': 0.0001,
 'model__verbose': 0,
 'model__warm_start': False}

In [12]:
pipe = pipe.set_params(model__C=0.1)

pipe.fit(X_train, y_train)

pipe.get_params()

{'memory': None,
 'steps': [('scaler', StandardScaler()),
  ('model', LogisticRegression(C=0.1, max_iter=200))],
 'transform_input': None,
 'verbose': False,
 'scaler': StandardScaler(),
 'model': LogisticRegression(C=0.1, max_iter=200),
 'scaler__copy': True,
 'scaler__with_mean': True,
 'scaler__with_std': True,
 'model__C': 0.1,
 'model__class_weight': None,
 'model__dual': False,
 'model__fit_intercept': True,
 'model__intercept_scaling': 1,
 'model__l1_ratio': 0.0,
 'model__max_iter': 200,
 'model__n_jobs': None,
 'model__penalty': 'deprecated',
 'model__random_state': None,
 'model__solver': 'lbfgs',
 'model__tol': 0.0001,
 'model__verbose': 0,
 'model__warm_start': False}

In [13]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__C": [0.01, 0.1, 1, 10]
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=5
)

grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV score:", grid.best_score_)

Best params: {'model__C': 10}
Best CV score: 0.9666666666666668


In [14]:
best_pipe = grid.best_estimator_

pred_best = best_pipe.predict(X_test)

print("Pred from best pipeline (first 5):", pred_best[:5])

Pred from best pipeline (first 5): [0 2 1 1 0]
